# LUMEN-299 · 4 — The book of c

**The failure.** Five experiments between 1928 and 1941, two kinds of instrument,
excellent agreement: Cochran's $Q$ says they are consistent with one value, and the pooled
interval is a few kilometres a second wide. It excludes the truth by three of its own
standard errors. Nothing in the five records, and nothing in any test of their agreement,
could have said so.

This notebook builds the corpus and pools it every way `axiom.meta` knows — fixed,
random, three-level, Bayesian with a party level — and shows which of those ways sees the
failure, which cannot, and what the corpus is entitled to tell the next experimenter.

| | reaches into |
|---|---|
| the corpus | `meta.StudyRecord`, `meta.Corpus`, `meta.record_from_summary` |
| the classical pools | `meta.fixed_effect`, `meta.random_effects`, `meta.heterogeneity`, `meta.prediction_interval` |
| the party level | `meta.three_level`, `meta.variance_shares` |
| influence and small-study effects | `meta.leave_one_out`, `meta.egger`, `meta.funnel_data` |
| the Bayesian pool with parties | `meta.pool`, `meta.PoolSpec`, `meta.PoolPriors` |
| the prior the next instrument deserves | `meta.prior_from_pool` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import pandas as pd

import lightspeed as L

from axiom.meta import (
    Corpus, Pooled, PoolPriors, PoolSpec, StudyRecord, ThreeLevel, egger, fixed_effect,
    funnel_data, heterogeneity, leave_one_out, pool, prediction_interval, prior_from_pool,
    random_effects, record_from_summary, three_level,
)

from axiom.display import enable, show, table

sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import AQUA, BLUE, MUTED, ORANGE, VIOLET, caption, compare, figure, intervals, mark_x

enable();  # every axiom result renders itself from here on

world = L.simulate()
FAMILIES = list(L.FAMILIES)


def record(r: L.Readout) -> StudyRecord:
    e = r.experiment
    return record_from_summary(
        study=e.key, contributor=e.family, quantity="relative_change",
        estimate=L.ppm(r.read_vacuo), se=r.se / L.C_TRUE * 1e6, read="experiment",
        family="speed-of-light", moderators={"year": float(e.year)}, period=str(e.year),
        source=e.who, detail={"medium": e.medium, "generation": e.generation},
    )


corpus = Corpus(records=tuple(record(r) for r in world.readouts), name="the book of c")
print(len(corpus), "records |", len(corpus.contributors()), "parties:", corpus.contributors())
km = lambda x: L.from_ppm(x)  # ppm from the defined value -> km/s
table([[r.period, r.source, r.contributor, f"{r.estimate:+,.1f}", f"{r.se:,.2f}", f"{km(r.estimate):,.1f}"]
       for r in corpus.records],
      headers=("year", "who", "party", "ppm from truth", "± ppm", "km/s"),
      title="every record is dimensionless — parts per million from the 1983 value — so nothing needs a transfer")

The quantity pooled is `relative_change`: parts per million from the defined value. That
is not a cheat — the pool would be identical in km/s — it is what makes every record
dimensionless so that `StudyRecord` accepts it without a licensed transfer, and it is
what the corpus would have been pooling in 1941 had it known the answer. It did not, and
the notebook keeps pretending it did not until the last section.

## 1 · Five experiments that agree

The bandwagon years: Karolus and Mittelstaedt's Kerr cell, the mile-long vacuum tube,
Anderson twice, Hüttel. Ask whether they agree with each other, then pool them.

In [ ]:
def subset(lo, hi):
    return Corpus(records=tuple(r for r in corpus.records if lo <= int(r.period) <= hi), name=f"{lo}-{hi}")

five = subset(1928, 1941)
y5, se5 = five.arrays()
het = heterogeneity(y5, se5)
fe5 = fixed_effect(y5, se5)
re5 = random_effects(y5, se5, tau_method="reml")
print(f"Q = {het.q:.2f} on {het.df} df, p = {het.p_value:.2f}, I² = {het.i2:.0%}: the five agree")
table([["fixed effect", f"{km(fe5.estimate):,.1f}", f"[{km(fe5.interval.lower):,.1f}, {km(fe5.interval.upper):,.1f}]",
        f"{(L.C_TRUE - km(fe5.estimate)) / (fe5.se * L.C_TRUE / 1e6):.1f}"],
       ["random effects (REML)", f"{km(re5.estimate):,.1f}", f"[{km(re5.interval.lower):,.1f}, {km(re5.interval.upper):,.1f}]",
        f"{(L.C_TRUE - km(re5.estimate)) / (re5.se * L.C_TRUE / 1e6):.1f}"],
       ["the truth", f"{L.C_TRUE:,.1f}", "", ""]],
      headers=("pool", "c (km/s)", "95% interval", "truth is this many se away"),
      title="1928–1941, five records")
print(f"tau between the five: {re5.tau:.1f} ppm — the random-effects pool is the fixed-effect pool")

In [ ]:
rows = [(f"{r.period} {r.source}", km(r.estimate), km(r.estimate - 1.96 * r.se), km(r.estimate + 1.96 * r.se))
        for r in five.records]
rows.append(("pooled, 1928–41", km(re5.estimate), km(re5.interval.lower), km(re5.interval.upper)))
fig = intervals(
    rows, ref=L.C_TRUE, ref_label="the truth", highlight="pooled, 1928–41",
    title="Five experiments that agree, and the value they agree on",
    subtitle="each ±1.96 se as reported; the pool is a few km/s wide and does not contain the truth",
    x_title="c (km/s)",
)
caption(fig, "Every reported error bar is an honest statement about scatter. The pooled "
             "bar inherits nothing but scatter, so it inherits none of what the five "
             "have in common — which is the only thing wrong with it.")

Cochran's $Q$ is the wrong question, asked correctly. It tests whether the five differ
from each other by more than their reported errors, and they do not; the systematic they
share is invisible to any test of their mutual agreement, and a pool that weights by the
reported errors quotes a precision made only of the part the instruments cannot share.

## 2 · Add the one that did not agree, and give the families a level

Michelson's 1926 result sits above the five. Put it in and the two-level pool still
excludes the truth, by a kilometre a second. Now give the corpus a *party level* — the instrument family, whose
systematics travel together — and ask `three_level` for two variance components instead
of one.

In [ ]:
def nested(c: Corpus) -> ThreeLevel:
    y, se = c.arrays()
    return three_level(y, se, list(range(len(c))), [FAMILIES.index(r.contributor) for r in c.records])

rows = []
for label, c in (("1928–41, five records, two families", five),
                 ("1926–41, six records, two families", subset(1926, 1941))):
    y, se = c.arrays()
    two = random_effects(y, se, tau_method="reml")
    three = nested(c)
    for name, est, lo, hi, tau_s, tau_p in (
        ("two-level REML", two.estimate, two.interval.lower, two.interval.upper, two.tau, None),
        ("three-level REML", three.estimate, three.interval.lower, three.interval.upper, three.tau_study, three.tau_party),
    ):
        rows.append([label, name, f"{km(est):,.1f}", f"[{km(lo):,.1f}, {km(hi):,.1f}]",
                     f"{tau_s:.0f}", "—" if tau_p is None else f"{tau_p:.0f}",
                     "yes" if lo <= 0.0 <= hi else "NO"])
table(rows, headers=("records", "pool", "c (km/s)", "95% interval", "tau within (ppm)",
                     "tau between families", "covers the truth"))

Two rows are the argument.

With 1926 in, the family level *does* rescue the pool: the rotating-mirror family now
disagrees with the Kerr cells, `three_level` reads that disagreement as a between-family
component of about thirty parts per million, the interval on the mean widens from twenty
to thirty kilometres a second — what the two families actually support — and it covers
the truth. That is what a party level is for —
a systematic shared within a family is a variance component, and counting six records
from two families as six draws quotes a precision the book does not have.

Without 1926 it does not. Both families carried the same expectation, so there is no
disagreement between them for the family level to read, and the three-level interval
excludes the truth as confidently as the two-level one. **No hierarchy over parties can
see a systematic every party shares.** The only thing that found it, historically, was an
instrument from outside both families, and the value of that instrument to the programme
is not visible in any pooling of the ones before it.

## 3 · The whole record before 1950

Twelve records, three families, a century apart in quality. The heterogeneity is no
longer subtle — two thirds of the variance at a typical record is the spread between
generations *within* a family — and three more instruments say what a pool of them may
honestly hand the next experimenter.

In [ ]:
before = subset(1849, 1949)
y12, se12 = before.arrays()
het12 = heterogeneity(y12, se12)
re12 = random_effects(y12, se12, tau_method="reml")
pi12 = prediction_interval(re12)
three12 = nested(before)
print(f"I² = {het12.i2:.1%} across the twelve; tau = {re12.tau:.0f} ppm between records")
table([["random effects, mean", f"{km(re12.estimate):,.0f}", f"[{km(re12.interval.lower):,.0f}, {km(re12.interval.upper):,.0f}]",
        "yes" if re12.interval.contains(0.0) else "NO"],
       ["prediction interval for a new record", f"{km(re12.estimate):,.0f}", f"[{km(pi12.lower):,.0f}, {km(pi12.upper):,.0f}]",
        "yes" if pi12.contains(0.0) else "NO"],
       ["three-level, mean", f"{km(three12.estimate):,.0f}", f"[{km(three12.interval.lower):,.0f}, {km(three12.interval.upper):,.0f}]",
        "yes" if three12.interval.contains(0.0) else "NO"]],
      headers=("what", "c (km/s)", "95% interval", "covers the truth"), title="1849–1941, twelve records")
shares = three12.shares(float(np.median(se12**2)))
print(f"at the median record's sampling variance the split is — sampling {shares[0]:.0%}, "
      f"within family {shares[1]:.0%}, between families {shares[2]:.0%}")

In [ ]:
loo = leave_one_out(y12, se12, method="reml")
order = np.argsort(-np.abs(loo.influence))[:4]
table([[before.records[i].period, before.records[i].source, f"{km(loo.estimates[i]):,.0f}", f"{loo.influence[i]:+.2f}"]
       for i in order],
      headers=("drop", "who", "pooled c without it", "shift (pooled se)"),
      title="the four most influential records")

test = egger(y12, se12)
print(f"\nEgger: intercept {test.intercept:+.2f} ± {test.se:.2f}, p = {test.p:.3f} — "
      + ("the precise records sit systematically to one side" if test.p < 0.05 else "no small-study effect"))

In [ ]:
funnel = funnel_data(y12, se12, re12)
fig = figure(
    title="The funnel, before 1950",
    subtitle="each record against its standard error; the contours are where a record with no systematic would fall",
    x_title="ppm from the defined value", y_title="standard error (ppm)",
)
for contour in funnel.contours:
    fig.add_scatter(x=list(contour.lower) + list(contour.upper)[::-1],
                    y=list(contour.se) + list(contour.se)[::-1], mode="lines",
                    line={"color": MUTED, "width": 1, "dash": "dot"}, showlegend=False, hoverinfo="skip")
for family, color in L.FAMILY_COLOR.items():
    rows = [(r.estimate, r.se, r.source) for r in before.records if r.contributor == family]
    if rows:
        fig.add_scatter(x=[r[0] for r in rows], y=[r[1] for r in rows], mode="markers",
                        marker={"size": 10, "color": color}, name=L.FAMILY_LABEL[family],
                        text=[r[2] for r in rows], hovertemplate="%{text}: %{x:+.0f} ppm ± %{y:.0f}")
mark_x(fig, 0.0, text="the truth")
fig.update_yaxes(type="log", autorange="reversed")
fig.update_xaxes(range=[-1500, 1500])
caption(fig, "Read it from the bottom. The most precise records — the Kerr cells and the vacuum "
             "tube — sit to the left of the truth together, which is what a shared "
             "systematic looks like in a funnel and what Egger's test is detecting. The "
             "wide early records scatter on both sides as noise should.")

## 4 · The Bayesian pool, with and without a party level

Back to the six records of 1926–41, where the party level is the whole question.
`meta.pool` fits the same tree with a sampler-free Laplace backend: `effect_key="study"`
gives every record its own effect under one `tau`; `effect_key="nested"` puts the family
above the studies. The party-level spread is fixed at the REML value from section 2,
which is what lets Laplace fit the tree exactly (note 0029) and is a defensible choice
with two parties in any case.

In [ ]:
six = subset(1926, 1941)
three6 = nested(six)
priors = PoolPriors(mu_scale=300.0, tau_scale=100.0, tau_party_fixed=max(three6.tau_party, 1.0))
fits = {}
for key in ("study", "nested"):
    out = pool(PoolSpec(family="speed-of-light", effect_key=key, priors=priors, mass=0.95),
               six, backend="laplace", draws=4000, seed=1)
    assert isinstance(out, Pooled)
    fits[key] = out.result
table([[key, f"{r.n_effects} effects / {r.n_parties or '—'} parties", f"{km(r.mu.mean):,.1f}",
        f"[{km(r.mu.interval.lower):,.1f}, {km(r.mu.interval.upper):,.1f}]",
        f"{(r.mu.interval.upper - r.mu.interval.lower) * L.C_TRUE / 1e6:.1f}", f"{r.tau.mean:.0f}",
        "—" if r.tau_party is None else f"{r.tau_party.mean:.0f}",
        "yes" if r.mu.interval.contains(0.0) else "NO"]
       for key, r in fits.items()],
      headers=("effect_key", "levels", "mu (km/s)", "95% interval", "width", "tau", "tau_party", "covers the truth"),
      title="the same six records, two beliefs about what is independent")
table([[a.name.split("[")[1][:-1], f"{km(a.mean):,.0f}", f"[{km(a.interval.lower):,.0f}, {km(a.interval.upper):,.0f}]"]
       for a in fits["nested"].alphas],
      headers=("family", "alpha (km/s)", "95% interval"), title="what each family's own mean came out at")

## 5 · The prior the next instrument deserves

The Bayesian `study` pool lands where the REML one did, its prior on `tau` buying it a
little width and its upper edge still stopping a few hundred metres a second short of
the truth; the `nested` pool is half again as wide and covers it comfortably, and the two family means it reports are ten
kilometres a second apart — the disagreement the party level is built from.

In 1950 Essen measured $c$ with a cavity resonator: a new family. What prior should the
book of 1926–41 have handed him? `prior_from_pool` gives four answers. The mean's own
interval is the narrowest and the wrongest. The party-level pair is the one to read: a
prior for *another Kerr cell* is centred on the Kerr cells' own mean and inherits their
systematic as if it were knowledge; a prior for a *new party* is centred on the family
mean and widened by the spread between families.

In [ ]:
rows = []
for target, party in (("mu", ""), ("predictive", ""), ("same_party", "kerr-cell"), ("new_party", "")):
    prior, line = prior_from_pool(fits["nested"], target=target, party=party)
    m, s = prior.hyper["mu"], prior.hyper["sigma"]
    rows.append([target, party or "—", f"{km(m):,.0f}", f"±{s * L.C_TRUE / 1e6:,.0f}",
                 f"{(0.0 - m) / s:+.1f}"])
table(rows, headers=("target", "party", "prior mean (km/s)", "prior sd (km/s)", "truth, in prior sds"),
      title="what the book would have told Essen in 1950")
essen = world["essen_1950"]
print(f"Essen found {essen.read:,.1f} ± {essen.se:.1f}; the truth is {L.C_TRUE:,.1f}")
print("the twelve-record book before 1950 hands on the same lesson with wider numbers:")
big = pool(PoolSpec(family="speed-of-light", effect_key="nested",
                    priors=PoolPriors(mu_scale=500.0, tau_scale=300.0, tau_party_fixed=max(three12.tau_party, 1.0))),
           before, backend="laplace", draws=4000, seed=1)
assert isinstance(big, Pooled)
for target in ("mu", "new_party"):
    prior, _ = prior_from_pool(big.result, target=target)
    print(f"   {target:10s} {km(prior.hyper['mu']):,.0f} ± {prior.hyper['sigma'] * L.C_TRUE / 1e6:,.0f} km/s")

## 6 · After 1972

Add the four electromagnetic measurements and pool everything. The laser dominates by a
factor of a million in weight and the interval collapses onto it, which is correct and
uninteresting: the corpus has stopped being a pool and become one experiment with fifteen
footnotes. The `leave_one_out` influence table says so — drop Evenson and the pooled value
moves by more than its own standard error.

In [ ]:
y16, se16 = corpus.arrays()
re16 = random_effects(y16, se16, tau_method="reml")
loo16 = leave_one_out(y16, se16, method="reml")
print(f"all sixteen, random effects: {km(re16.estimate):,.4f} "
      f"[{km(re16.interval.lower):,.4f}, {km(re16.interval.upper):,.4f}] — truth {L.C_TRUE:,.4f}")
worst = int(np.argmax(np.abs(loo16.influence)))
print(f"most influential: {corpus.records[worst].source}, shift {loo16.influence[worst]:+.1f} pooled se when dropped")
print(f"the record with the largest weight carries {max(re16.weights) / sum(re16.weights):.1%} of the pool")

## What notebook 4 established

1. **Agreement is not accuracy.** Five records from 1928–41 pass Cochran's test and pool
   to an interval that excludes the truth by three standard errors. The test asks whether
   they differ from each other; nothing in it can ask what they share.
2. **A party level sees a shared systematic only when parties disagree.** With 1926 in the
   corpus the rotating mirrors and the Kerr cells disagree, `three_level` reads that as
   `tau_party`, and the interval covers the truth. Without it the two families agree on
   the same wrong number and no hierarchy can tell.
3. **Before 1950 the record is dominated by what generations of one family did not
   share**, Egger's test finds the precise records sitting to one side, and the honest
   number for a new experiment is the prediction interval — nearly two hundred
   kilometres a second wide — not the interval on the mean, which is sixty wide and wrong.
4. **The prior for a new kind of instrument is not the prior for another of the same
   kind.** From the 1926–41 book, `target="mu"` hands Essen ±8 km/s with the truth 1.3
   standard deviations out; `same_party` for a Kerr cell is centred on the Kerr cells'
   own systematic; `new_party` is ±17 with the truth well inside. The one that looks most
   informative is the one that is wrong.

Notebook 5 asks what the programme would have decided, one experiment at a time.